# Before and After: Plotting Floods and Other Changes
### Visualize Normalized Difference Indices, Make Binary Masks, & Calculate Gain and Loss

Cleo Young | cyoung@umn.edu


## 1. Design Framework

### 1.1. Purpose and Description 
Due to climate change, extreme flooding is becoming increasingly common. Quick and automated mapping of flood extent is necessary for prompt disaster response and recovery efforts, and with multiple methods  available for doing so, it's helpful to be able to quickly compare the results of different indices and thresholds. 

This tool allows users to choose between 5 different normalized difference indices, all of which may be used in flood detection or other land change detection. These are calculated using values from different bands of Sentinel-2 multispectral satellite imagery. The generic formula is (first band - second band) / (first band + second band). This produces a number between -1 and 1, with values closer to 1 indicating a stronger likelihood of the target spectral characteristic. 

Indices like the the Normalized Difference Water Index (NDWI) and the Modified Normalized Difference Water Index (MNDWI) can be used to calculate water extent and, through change detection, flooding extent. Indices like NDVI, NDBI, and NDTI can be used to provide context about the vegetation, urban build-up, and water turbidity of the area of interest, which can then be turned into masks and compared with the water extent masks to filter out noise, understand the impacts of flooding, or gain other contextual insight. 


### 1.2 Scenario
Frankfort, Kentucky experienced extreme flooding in early April 2025. At least 22 people died and many communities are facing difficult decisions amidst great destruction. In November 2024, the previous fall, the Kentucky River saw extremely low water levels. These indices provide a way to get more information about how the natural and built environment has responded to these climatic extremes. 

### 1.3 Goals

My primary goal was to show that I can take a fairly limited tool designed for a specific purpose (my MVP) and refactor it to be modular and expanadable, with a wider range of features and potential uses. This tool can now be used to compare many different spectral aspects of imagery from two dates. 

### 1.4 Feature List
- Read two multi-band rasters
- Check to make sure they're aligned
- Take user input about which index to use, while providing tips about what each does
- Take user input about which bands hold the relevant spectral data
- Calculate and visualize the chosen index for both rasters
- Ask the user to choose a threshold between 0 and 1 to create a binary mask (water vs. non-water or something else) 
- Calculate and visualize the resulting binary mask
- If they aren't happy with it, the can change the threshold slider and visualize it again 
- Calculate and visualize the change between the "before" and "after" binary masks
- Report the total areas of gain and loss in square meters
- Allow the user to save the change mask as a new raster file

___

## 2. Code

## 2.1. Setup & Installation

In [ ]:
# Modules and packages

# For doing math with rasters
import numpy as np
import rasterio
# For plotting
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.patches as mpatches
# For the UI 
import ipywidgets as widgets
from ipywidgets import Dropdown, Text, Label, VBox, HBox, Output
from IPython.display import display, clear_output


## 2.2. Core Functions

### Data Handling and Preparation  
These functions read and write the raster files.  
Crucially, because we're doing change detection, we need the rasters to be aligned. In a future version of this notebook we'll have a function to align them, but for now, we'll have to just settle for error handling. 

In [ ]:
# Read a raster from a file path
def read_raster(filepath):
    # open and auto-close the file, store as object named "src" cuz that's the default i guess
    try:
        with rasterio.open(filepath) as src:
            # pixel values in a NumPy array
            input_array = src.read()
            # get the metadata as a profile
            profile = src.profile
        # Returns a tuple 
        return input_array, profile
    except Exception as e:
        raise IOError(f"Failed to read raster at {filepath}: {e}")

In [335]:
# Save the output water extent binary mask
def write_raster(output_array, reference_profile, output_path):
    # Copy the original metadata profile and adjust it
    profile = reference_profile.copy()
    profile.update(
        dtype=rasterio.uint8,  # save as 8-bit unsigned integer (0 or 1)
        count=1,               # just one band (the mask)
    )
    
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(output_array.astype(rasterio.uint8), 1)  # write the mask as band 1

In [336]:
# Helper function to check raster alignment
def check_raster_alignment(data1, data2, profile1, profile2):
    # Check if shapes match
    if data1.shape[1:] != data2.shape[1:]:
        raise ValueError("Raster dimensions do not match.")
    
    # Check if transforms match
    if profile1['transform'] != profile2['transform']:
        raise ValueError("Raster transforms do not match.")
    
    # If both checks pass, return True
    return True

### Calculating and Visualizing Indices

Instead of different functions for each index, I decided to build this tool around the ability to calculate many different indices that all have the same basic equation.  
This will take input from widgets about which bands to use, but other than that, the user interface handles the distinction between indices.  
The resulting raster is visualized with a color ramp. 

In [337]:
# Calculates a normalized difference index
def compute_index(first_band, second_band, epsilon=1e-10):
    # This approach is far more modular than the conditional logic in my MVP
    # All the indices have the same equation, so it doesn't care which one 
    numerator = first_band.astype(float) - second_band.astype(float)
    denominator = first_band.astype(float) + second_band.astype(float) + epsilon
    return numerator / denominator

# Visualizes the results of the index calculation
def plot_index(index_array, title="Water Index", cmap='viridis'):
    """Displays a continuous index raster."""
    plt.figure(figsize=(8, 8))
    plt.imshow(index_array, cmap=cmap)
    plt.colorbar(label='Index Value')
    plt.title(title)
    plt.axis('off')
    plt.show()       

### Creating a Binary Mask  

These helper functions apply a threshold to the indexed raster.   
Everything above the threshold is classified as a 1 (True) and below, as a 0 (False).  
This is then visualized with matplotlib. 

In [338]:
# Creates a binary mask with water and non-water pixels
def apply_threshold(index_array, threshold):
    # Boolean comparison for every pixel
    return (index_array > threshold).astype(np.uint8)

# Plots the binary mask
def plot_mask(mask, title="Binary Mask", colors=['lightgreen', 'blue']):
    # Displays a classified binary mask.
    custom_cmap = ListedColormap(colors)
    plt.figure(figsize=(8, 8))
    plt.imshow(mask, cmap=custom_cmap, interpolation='none')
    plt.title(title)
    plt.axis('off')
    plt.show()

### Change Detection  
This part performs the important task of subtracting one raster from the other and visualizing the results.  
This is important because it allows comparison between two dates, accomplishing one of our biggest goals that was missing from the MVP. 

In [339]:
# subtract the two masks to do change detection 
def subtract_masks(mask_after, mask_before):
    # subracts the 'before' mask from the 'after' mask to detect changes
    # returns a single-band raster
    # 1 = gain, 0 = no change, -1 = loss
    mask_after = np.asarray(mask_after)
    mask_before = np.asarray(mask_before)

    return mask_after.astype(np.int8) - mask_before.astype(np.int8)

# Plot the result
def plot_change_mask(change_mask, title="Change Detection Results"):
    # Plots the change detection mask.
    #  1 (gain): blue
    # -1 (loss): brown
    #  0 (no change): light gray
    cmap = ListedColormap(['saddlebrown', 'lightgray', 'blue'])
    norm = BoundaryNorm([-1.5, -0.5, 0.5, 1.5], cmap.N)

    plt.figure(figsize=(8, 8))
    plt.imshow(change_mask, cmap=cmap, norm=norm, interpolation='none')
    plt.title(title)
    plt.axis('off')
    # Custom legend
    legend_patches = [
        mpatches.Patch(color='saddlebrown', label='Loss (-1)'),
        mpatches.Patch(color='lightgray', label='No Change (0)'),
        mpatches.Patch(color='blue', label='Gain (+1)')
    ]
    plt.legend(handles=legend_patches, loc='lower center', bbox_to_anchor=(0.5, -0.1), ncol=3, frameon=False)

    plt.tight_layout()
    plt.show()

### Calculating Statistics  
This function is used to give the user a little report of the area in square meters occupied by the detected change - this could be the extent of flooding, but it could also represent the extent of areas with vegetation loss, urban sprawl, etc. 

In [340]:
# Calculate the total area in square meters 
def calculate_area_m2(mask, transform):
    """Calculates total area of pixels == 1 in square meters."""
    pixel_area = abs(transform.a * transform.e)  # pixel width * pixel height
    return np.sum(mask == 1) * pixel_area

## 2.3 User Interface

### Output Areas  

This is one of two really important pieces that make the whole UI come to life. The other one is the Vbox layout at the very end of this section. 

In [341]:
# gives info about the file
file_output = widgets.Output()

# shows descriptions of the indices
index_output = widgets.Output()
# two area to preview both indices side by side
index_plot_before = widgets.Output()
index_plot_after = widgets.Output()

# areas for visualizing the binary masks 
mask_plot_before = widgets.Output()
mask_plot_after = widgets.Output()

# Change mask output areas for visualizing and saving
change_mask_plot = widgets.Output()
change_save_output = widgets.Output()

### Load two rasters

In [342]:
# Input file path widgets for "before" and "after" rasters
before_label = widgets.Label("Enter path to 'Before' raster:")
before_path = widgets.Text(placeholder="e.g., data_before.tif")
after_label = widgets.Label("Enter path to 'After' raster:")
after_path = widgets.Text(placeholder="e.g., data_after.tif")

# Button to trigger load function
load_button = widgets.Button(
    description="Load Rasters", 
    button_style='success',
    layout=widgets.Layout(width='200px')
)

# Global variables to store the two rasters 
raster_data_before = {}
raster_data_after = {}

# Button callback function for loading the rasters
def load_rasters(b):
    with file_output:
        clear_output()
        try:
            # Read rasters
            data1, profile1 = read_raster(before_path.value)
            data2, profile2 = read_raster(after_path.value)

            # Check alignment using the helper function
            check_raster_alignment(data1, data2, profile1, profile2)

            # Store aligned rasters in the global dictionary
            raster_data_before['array'], raster_data_before['profile'] = data1, profile1
            raster_data_after['array'], raster_data_after['profile'] = data2, profile2

            # Inform user that the rasters were loaded and aligned successfully
            print("✅ Rasters loaded and aligned successfully. You can now select an index.")

        except Exception as e:
            print(f"❌ Failed to load rasters: {e}")

load_button.on_click(load_rasters)

### Index Selection and Band Input

In [343]:
# Index selection dictionary

INDEX_CONFIG = {
    "NDVI": {
        "label_1": "NIR Band:",
        "label_2": "Red Band:",
        "description": (
            "NDVI (Normalized Difference Vegetation Index) highlights healthy vegetation. "
            "High values indicate dense, green plant life. "
            "Equation: (NIR - Red) / (NIR + Red)"
        )
    },
    "NDWI": {
        "label_1": "Green Band:",
        "label_2": "NIR Band:",
        "description": (
            "NDWI (Normalized Difference Water Index) emphasizes water bodies. Useful for surface water mapping. "
            "Equation: (Green - NIR) / (Green + NIR)"
        )
    },
    "MNDWI": {
        "label_1": "Green Band:",
        "label_2": "SWIR Band:",
        "description": (
            "MNDWI (Modified NDWI) improves water detection in built-up areas by replacing NIR with SWIR. "
            "Equation: (Green - SWIR) / (Green + SWIR)"
        )
    },
    "NDBI": {
        "label_1": "SWIR Band:",
        "label_2": "NIR Band:",
        "description": (
            "NDBI (Normalized Difference Built-up Index) identifies urban or built-up areas. "
            "High values often correspond to impervious surfaces. "
            "Equation: (SWIR - NIR) / (SWIR + NIR)"
        )
    },
    "NDTI": {
        "label_1": "Red Band:",
        "label_2": "Green Band:",
        "description": (
            "NDTI (Normalized Difference Turbidity Index) highlights turbidity in water bodies. "
            "Higher values indicate greater suspended sediment or turbidity. "
            "Equation: (Red - Green) / (Red + Green)"
        )
    }
}

In [344]:
# Tell the user what's up
index_hint = widgets.HTML(
    value="<i>Please select an index to use for change detection.</i>",
)

# Dropdown for selecting index
# This now uses a dictionary so we can easily add or remove indices
index_dropdown = Dropdown(
    # the keys of the dictionary from previous cell
    options=list(INDEX_CONFIG.keys()),
    description="Index:",
    value="MNDWI",  # Default value
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

# The new approach to index selection allows me to offer many indices without needing widgets for all of them!

# Widgets for band selection (description will be set dynamically)
band1_input = Text(description="", layout=widgets.Layout(width='200px'))
band2_input = Text(description="", layout=widgets.Layout(width='200px'))

# Output widget to display index description
index_description = widgets.Output(layout={'width': '700px'})

# shows the correct 2 band input widget labels depending on choice of index
# and the description
def update_band_labels(change):
    index_key = change['new']
    config = INDEX_CONFIG[index_key]
    
    band1_input.description = config['label_1']
    band2_input.description = config['label_2']
    
    with index_description:
        index_description.clear_output()
        print(config['description'])

# Bind update function
index_dropdown.observe(update_band_labels, names='value')

In [345]:
# Button to calculate and visualize the index
calc_index_button = widgets.Button(
    description="Calculate Index",
    button_style='success',
    layout=widgets.Layout(width='200px')
)

# BUTTON CALLBACK FUNCTION
# Calculates and visualizes index
def on_calc_index_button_click(b):
    index_plot_before.clear_output()
    index_plot_after.clear_output()

    try:
        # Convert input to integers
        band1_idx = int(band1_input.value)
        band2_idx = int(band2_input.value)

        if band1_idx < 1 or band2_idx < 1:
            raise ValueError("Band indices must be greater than zero.")

        # Adjust for zero-based indexing
        band1_before = raster_data_before['array'][band1_idx - 1]
        band2_before = raster_data_before['array'][band2_idx - 1]

        band1_after = raster_data_after['array'][band1_idx - 1]
        band2_after = raster_data_after['array'][band2_idx - 1]

        # Compute index for before and after
        index_before = compute_index(band1_before, band2_before)
        index_after = compute_index(band1_after, band2_after)

        # Plot both
        with index_plot_before:
            plot_index(index_before, title="Index (Before)")

        with index_plot_after:
            plot_index(index_after, title="Index (After)")

    except Exception as e:
        print(f"Error calculating the index: {e}")

calc_index_button.on_click(on_calc_index_button_click)

In [346]:
# Index UI

# Call once to initialize
update_band_labels({'new': index_dropdown.value})

index_ui = VBox([
    index_hint,
    index_dropdown,
    HBox([band1_input, band2_input]),
    index_description,
    index_output,
    calc_index_button,
    HBox([index_plot_before, index_plot_after])  # side-by-side plots
])

### Creating the Binary Mask

In [347]:
# Threshold slider
threshold_input = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=0.5,
    step=0.01,
    description='Threshold:',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# Button to apply threshold
apply_threshold_button = widgets.Button(
    description="Apply Threshold to Both",
    button_style='success',
    layout=widgets.Layout(width='200px')
)

# Tell the user it's dynamic!
threshold_hint = widgets.HTML(
    value="<i>You can adjust the threshold and visualize again before moving on.</i>",
)

# Store output masks for later
binary_masks = {
    "before": None,
    "after": None
}

# BUTTON CALLBACK for applying the threshold and visualizing masks
def on_apply_threshold_click(b):
    mask_plot_before.clear_output()
    mask_plot_after.clear_output()

    try:
        threshold = threshold_input.value

        # Same band inputs as before
        band1_idx = int(band1_input.value)
        band2_idx = int(band2_input.value)

        band1_before = raster_data_before['array'][band1_idx - 1]
        band2_before = raster_data_before['array'][band2_idx - 1]
        band1_after = raster_data_after['array'][band1_idx - 1]
        band2_after = raster_data_after['array'][band2_idx - 1]

        # Compute indices
        index_before = compute_index(band1_before, band2_before)
        index_after = compute_index(band1_after, band2_after)

        # Apply threshold to generate binary masks
        mask_before = apply_threshold(index_before, threshold)
        mask_after = apply_threshold(index_after, threshold)

        # Store for later use (optional)
        binary_masks["before"] = mask_before
        binary_masks["after"] = mask_after

        # Plot both masks
        with mask_plot_before:
            plot_mask(mask_before, title="Binary Mask - Before")

        with mask_plot_after:
            plot_mask(mask_after, title="Binary Mask - After")

    except Exception as e:
        print(f"Error applying threshold: {e}")
        
# Connect callback to button 
apply_threshold_button.on_click(on_apply_threshold_click)

In [348]:
# Mask UI
mask_ui = VBox([
    threshold_input,
    apply_threshold_button,
    threshold_hint,
    HBox([mask_plot_before, mask_plot_after])
])

### Change Detection

In [349]:
# create a button for it 
change_button = widgets.Button(
    description="Detect Change",
    button_style='success',
    layout=widgets.Layout(width='200px')
)

# CALLBACK FUNCTION for change detection button
def on_change_button_click(b):
    with change_mask_plot:  # Make sure this is the correct widget name
        change_mask_plot.clear_output()
        try:
            # Retrieve masks from dictionary
            mask_before = binary_masks.get("before")
            mask_after = binary_masks.get("after")

            if mask_before is None or mask_after is None:
                print("Binary masks not found. Please apply the threshold first.")
                return

            # Calculate difference mask
            change_mask = subtract_masks(mask_after, mask_before)

            # Store or use the result
            binary_masks["change"] = change_mask  # Optional, for saving later

            plot_change_mask(change_mask)
            
            # Calculate area of water gain (pixels == 1)
            gain_area_m2 = calculate_area_m2(change_mask, transform)

            # Calculate area of water loss (pixels == -1)
            loss_area_m2 = calculate_area_m2(change_mask == -1, transform)

            # Display areas
            print(f"🌊 Water Gain Area: {gain_area_m2:,.2f} square meters")
            print(f"🌍 Water Loss Area: {loss_area_m2:,.2f} square meters")
            
        except Exception as e:
            print(f"Error during change detection: {e}")
            
# Attach button to callback
change_button.on_click(on_change_button_click)

### Saving the Results

In [350]:
# Save the files

# Output file path
change_output_path = widgets.Text(
    value='change_mask.tif',
    description='Save As:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# A button to save the file
save_change_button = widgets.Button(
    description="Save Mask",
    button_style='success',
    layout=widgets.Layout(width='200px')
)

# BUTTON CALLBACK for saving 
def on_save_change_button_click(b):
    with change_save_output:
        change_save_output.clear_output()
        try:
            output_path = change_output_path.value
            if not output_path.endswith(".tif"):
                raise ValueError("Output filename must end with .tif")

            change_mask = binary_masks.get("change")
            if change_mask is None:
                raise ValueError("Change mask not found. Please run change detection first.")

            reference_profile = raster_data_before['profile']
            write_raster(change_mask, reference_profile, output_path)
            print(f"✅ Change mask saved to: {output_path}")
        except Exception as e:
            print(f"❌ Error saving file: {e}")
        
# Attach button to callback
save_change_button.on_click(on_save_change_button_click)

In [351]:
# change UI

change_ui = VBox([
    change_button,
    change_mask_plot,
    change_output_path,
    save_change_button,
    change_save_output
])

### Full UI Layout

In [352]:
# Combine all UI parts into the final layout
full_ui = VBox([
    HBox([
        VBox([before_label, before_path]),
        VBox([after_label, after_path])
    ]),  # raster file path inputs
    load_button,
    file_output,
    # It was at this point that I got introduced to the idea of nest Vboxes
    # If I could go back I'd do this more but I'm out of time! 
    index_ui,
    mask_ui,
    change_ui
])

## 3. Execution

**IMPORTANT!**  
The test rasters provided with this project, KY_Nov2024.tif and KY_April2025.tif, have the following bands:  
1. Blue
2. Green
3. Red
4. NIR
5. SWIR

They are composites made from Sentinel-2 images of Frankfort, Kentucky, on November 17th 2024 (not flooded) and April 8th 2025 (severely flooded).

In [353]:
display(full_ui)

# KY_Nov2024.tif
# KY_April2025.tif